In [1]:
from sklearn.metrics import ConfusionMatrixDisplay

from xaikd import datasets, models, utils

from matplotlib import pyplot as plt

import numpy as np 

from tqdm import tqdm

/home/pat/projects/xai-kd/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def confusion_matrix(
    arr_teachers,
):
    dataset_name = 'imagenet-cat'
    
    ncols = len(arr_teachers)
    
    dataset = datasets.construct(dataset_name)
    arr_spurious_types = dict(
        clean=0,
        copyright=1,
        watermark=2,
        jpeg=3
    )
    
    ds = dataset.create_subset(train_split=False)
  
    
    dl = datasets.build_dataloader(ds, shuffle=False)

        
    def get_prediction(model):
        arr_ypred = []
        arr_ytrue = []
        for x, y in dl:
            logits = model(x)
            ypred = np.argmax(logits.detach().cpu().numpy(), axis=1).tolist()
            ytrue = y.detach().numpy().tolist()
            arr_ypred.extend(ypred)
            arr_ytrue.extend(ytrue)
        return arr_ytrue, arr_ypred
    
    
    plt.figure(figsize=(2.5*ncols, 2.5))
    plt.suptitle(f"{dataset_name}", y=1.05)
    

    
    for tix, teacher in tqdm(enumerate(arr_teachers)):
        ax = plt.subplot(1, ncols, tix + 1)
        plt.title(teacher)
        model =  models.get_trained_model(teacher)

        utils.modify_last_layer_for_subclasses(model, dataset.selected_classes)


    
        arr_ytrue, arr_ypred = get_prediction(model)

        ConfusionMatrixDisplay.from_predictions(arr_ytrue, arr_ypred, ax=ax, colorbar=False)
        if tix > 0:
            plt.yticks([]); plt.ylabel("")
    
confusion_matrix(
    arr_teachers=[
        "imagenet-resnet18-tv",
        "imagenet-resnet50-tv",
        "imagenet-vgg16-tv",
        "imagenet-nfnetf0-dm",
#         "imagenet-vitb-tv",
    ]
    )

We have 350 images in classes [281, 282, 283, 284, 285, 286, 287]


Preparing `ImageNetSuperClass[[281, 282, 283, 284, 285, 286, 287]]` samples: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 350/350 [00:00<00:00, 2158832.94it/s]
3it [01:31, 36.86s/it]